In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,mean_squared_error

In [2]:
df = pd.read_csv("emails.csv")
df.head()


,Email No.,the,to,ect,and,for,of,a,you,hou,...,connevey,jay,valued,lay,infrastructure,military,allowing,ff,dry,Prediction
0,Email 1,0,0,1,0,0,0,2,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Email 2,8,13,24,6,6,2,102,1,27,...,0,0,0,0,0,0,0,1,0,0
2,Email 3,0,0,1,0,0,0,8,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Email 4,0,5,22,0,5,1,51,2,10,...,0,0,0,0,0,0,0,0,0,0
4,Email 5,7,6,17,1,5,2,57,0,9,...,0,0,0,0,0,0,0,1,0,0


In [3]:
df['Email No.'] = df['Email No.'].astype(str).str.extract(r'(\d+)').astype(int)

In [4]:
X = df.drop(columns=["Prediction"])
y= df["Prediction"]

In [5]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [6]:
# model = RandomForestClassifier(
#     n_estimators=200,
#     max_depth=10,
#     min_samples_split=5,
#     min_samples_leaf=2,
#     max_features='sqrt',
#     random_state=42
# )
model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=int((y==0).sum()/(y==1).sum()),
    reg_alpha=1,
    reg_lambda=1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train,y_train)

XGBClassifier(base_score=0.5, booster='gbtree', callbacks=None,
              colsample_bylevel=1, colsample_bynode=1, colsample_bytree=0.8,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', gamma=0, gpu_id=-1,
              grow_policy='depthwise', importance_type=None,
              interaction_constraints='', learning_rate=0.05, max_bin=256,
              max_cat_to_onehot=4, max_delta_step=0, max_depth=4, max_leaves=0,
              min_child_weight=1, missing=nan, monotone_constraints='()',
              n_estimators=150, n_jobs=0, num_parallel_tree=1, predictor='auto',
              random_state=42, reg_alpha=1, reg_lambda=1, ...)

In [7]:
# calculate train accuracy
y_train_pred = model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)
print("✅ Train Accuracy:", train_accuracy)

✅ Train Accuracy: 0.970510031423737


In [8]:
y_pred = model.predict(X_test)

In [9]:
print("Accuracy score : ",accuracy_score(y_test,y_pred))
print("RMSE : ",np.sqrt(mean_squared_error(y_test, y_pred)))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy score :  0.9555555555555556
RMSE :  0.21081851067789195

Confusion Matrix:
 [[695  44]
 [  2 294]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.94      0.97       739
           1       0.87      0.99      0.93       296

    accuracy                           0.96      1035
   macro avg       0.93      0.97      0.95      1035
weighted avg       0.96      0.96      0.96      1035



In [12]:
# Take user input
user_email = input("Enter email text (words matching your numeric columns): ")

# 7️Convert user input to numeric features
user_features = dict.fromkeys(X.columns, 0)
words = user_email.lower().split()
for word in words:
    if word in user_features:
        user_features[word] += 1

# Convert to DataFrame
X_new = pd.DataFrame([user_features], columns=X.columns)

# 8️⃣ Predict
prediction = model.predict(X_new)
probabilities = model.predict_proba(X_new)

pred_label = 'spam' if prediction[0] == 1 else 'ham'
print("\n📧 Email Text:", user_email)
print("✅ Prediction:", pred_label)
print("🔢 Probabilities:", dict(zip(model.classes_, probabi

Enter email text (words matching your numeric columns):  congratulations you win 100 ruppes



📧 Email Text: congratulations you win 100 ruppes
✅ Prediction: spam
🔢 Probabilities: {0: 0.13124228, 1: 0.8687577}
